# BioVision — training the vehicle specialist on VehiDE

13,945 images, 36,081 instances, seven damage types, annotated to an insurance
company's claim standards.

**Runs on Kaggle or Colab; it detects which.** Cell 1 prints the platform it
found and where it will write. Run the cells in order.

Before you start, two settings that live outside this notebook:

* **Accelerator.** Right panel → **Session options** → set it to a GPU. Without
  one this trains on CPU and will not finish.
* **Internet.** Same panel. Kaggle defaults it to *off*, and the install cell
  needs it. A failure there looks like a broken package, not a missing toggle.

Two things about the notebook itself:

* **Nothing that matters is written to the runtime.** A session can end at any
  time and takes its filesystem with it. Weights and metrics go to Kaggle's
  persistent working directory, or to Drive on Colab — so a disconnect costs the
  time since the last epoch, not the whole run.
* **VehiDE's own train/val split is preserved.** Its validation set becomes our
  test set and is never trained on. Reshuffling everything would produce numbers
  that cannot be compared with anything else published on this dataset.


## 0. Reproducibility and settings

In [ ]:
SEED = 20260311

MODEL = "yolo11s-seg.pt"   # 139 ms/image on production CPU vs 483 for medium (measured)
IMGSZ = 640                # VehiDE averages 1.7M px, so this is a real downscale;
                           # 960 is the fallback if the scratch row disappoints
EPOCHS = 100
BATCH = 16

# The class order IS the contract with backend/src/biovision/models/specialists/
# vehicle_yolo.py::VEHIDE_CLASSES. The model emits integer ids; reordering this
# list silently relabels every prediction.
#
# Alphabetical, because the order has to be a rule rather than a habit. Seven
# classes, matching what VehiDE actually contains (ADR-026).
CLASSES = [
    "dent",
    "glass_shatter",
    "lamp_broken",
    "missing_part",
    "punctured",
    "scratch",
    "torn",
]

import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
print(f"seed={SEED}  model={MODEL}  imgsz={IMGSZ}  classes={len(CLASSES)}")


In [ ]:
!pip -q install ultralytics


## 1. Where are we, and where does the output go

Runs on **Kaggle** or **Colab**. The difference that matters is not the GPU, it
is what happens when the session ends:

| | Kaggle | Colab (free) |
|---|---|---|
| Session limit | 9 h | 12 h |
| Idle disconnect | — (runs in the background) | ~90 min |
| Browser must stay open | no | **yes** |
| Weekly GPU | 30 h, stated | 15–30 h, variable |
| This dataset | already there, no download | 2.3 GB to fetch |

On Kaggle the output directory survives the session and is downloadable. On
Colab it does not, so weights go to Drive instead — a checkpoint under
`/content` dies with the runtime.


In [ ]:
from pathlib import Path

try:
    import kaggle_web_client  # noqa: F401

    PLATFORM = "kaggle"
except ImportError:
    PLATFORM = "colab" if Path("/content").exists() else "local"

if PLATFORM == "kaggle":
    # /kaggle/working persists for the session and is downloadable afterwards.
    RUNS = Path("/kaggle/working/runs")
    WORK = Path("/kaggle/temp/biovision")
elif PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    # Drive, not /content: a Colab runtime takes its filesystem with it.
    RUNS = Path("/content/drive/MyDrive/biovision/runs")
    WORK = Path("/content/biovision")
else:
    RUNS = Path("./runs")
    WORK = Path("./work")

RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

print(f"platform: {PLATFORM}")
print(f"weights  -> {RUNS}")
print(f"scratch  -> {WORK}")

import subprocess

print()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "NO GPU -- switch the accelerator on before training")


## 2. Point at VehiDE

**On Kaggle:** add the dataset to the notebook (right panel → Add Input → search
`vehide-dataset-automatic-vehicle-damage-detection`). Nothing is downloaded; it
is mounted read-only.

**On Colab:** the cell downloads it from Kaggle, which is far faster than
uploading 2.3 GB from home. It asks for a `kaggle.json`
(kaggle.com → Settings → API → Create New Token). If you would rather upload
once, put the extracted dataset at `MyDrive/datasets/vehide/` and set
`COLAB_SOURCE = "drive"`.


In [ ]:
import json
from pathlib import Path

COLAB_SOURCE = "kaggle"   # or "drive"

if PLATFORM == "kaggle":
    ROOT = Path("/kaggle/input")
elif PLATFORM == "colab" and COLAB_SOURCE == "kaggle":
    from google.colab import files

    print("Upload kaggle.json ...")
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle
    !kaggle datasets download -d hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection -p /content/vehide_zip
    !unzip -q -o /content/vehide_zip/*.zip -d /content/vehide
    ROOT = Path("/content/vehide")
else:
    ROOT = Path("/content/drive/MyDrive/datasets/vehide")

assert ROOT.exists(), f"{ROOT} does not exist"


def locate(root: Path):
    """Find the annotations and the directory each one's images live in.

    **Nothing here assumes a directory name.** Two attempts at hard-coding the
    layout were both wrong: the Kaggle mount is called `datasets`, not
    `vehide-...`, and the nesting inside it differs from the zip. Names are what
    the hosting platform chose; contents are what the dataset actually is.

    So: find the annotation files anywhere under the root, read the first
    filename out of each, and find the directory that holds it. That works
    whatever anyone called the folders.
    """
    annotations = sorted(root.rglob("*via_annos.json"))
    assert annotations, (
        f"no *via_annos.json anywhere under {root}.\n"
        f"Top level: {[e.name for e in sorted(root.iterdir())][:20]}"
    )

    # Index every directory that holds jpgs, so a filename can be traced back.
    by_name = {}
    for image in root.rglob("*.jpg"):
        by_name.setdefault(image.name, image.parent)

    found = {}
    for path in annotations:
        entries = json.loads(path.read_text())
        first = next(iter(entries.values()))
        filename = first.get("name") or first.get("filename")
        directory = by_name.get(filename)
        assert directory is not None, (
            f"{path.name} references {filename!r}, which is not among the "
            f"{len(by_name):,} images found under {root}"
        )
        kind = "val" if "val" in path.name.lower() else "train"
        found[kind] = (path, directory, len(entries))

    assert {"train", "val"} <= set(found), f"found only: {sorted(found)}"
    return found


located = locate(ROOT)
(TRAIN_ANNOS, TRAIN_IMAGES, n_train_entries) = located["train"]
(VAL_ANNOS, VAL_IMAGES, n_val_entries) = located["val"]

print(f"root:   {ROOT}")
print(f"train:  {TRAIN_ANNOS.name} -> {TRAIN_IMAGES}")
print(f"        {n_train_entries:,} annotated, {len(list(TRAIN_IMAGES.glob('*.jpg'))):,} images on disk")
print(f"val:    {VAL_ANNOS.name} -> {VAL_IMAGES}")
print(f"        {n_val_entries:,} annotated, {len(list(VAL_IMAGES.glob('*.jpg'))):,} images on disk")

assert n_train_entries + n_val_entries == 13945, (
    f"expected VehiDE's 13,945 annotated images, found "
    f"{n_train_entries + n_val_entries:,} -- is this the right dataset?"
)


## 3. The converter

VehiDE ships VGG Image Annotator JSON, not COCO — and not standard VIA either.
Its regions are flat: `{"all_x": [...], "all_y": [...], "class": "tray_son"}`,
with no `shape_attributes` or `region_attributes`.

Reading it as standard VIA parses without error and finds **nothing**. The first
inspection run reported 36,081 regions and zero classes. Both shapes are
accepted below, and `inspect_via` prints which one it found before anything is
converted.


In [ ]:
import json
from collections import Counter


def normalise(name: str) -> str:
    """Class names as YOLO will see them.

    Defined here rather than imported from a COCO cell that no longer exists.
    Its absence was a NameError on the first call to inspect_via -- caught by
    rehearsing the notebook against the real dataset instead of trusting that
    moving cells around preserved what they depended on.
    """
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def _region_class(region: dict) -> str | None:
    """Pull the class name out of a VIA region.

    Two shapes are accepted, because VehiDE does not write standard VIA:

    * VehiDE:  {"all_x": [...], "all_y": [...], "class": "tray_son"}
    * VIA:     {"shape_attributes": {...}, "region_attributes": {"damage": "..."}}

    Reading VehiDE as standard VIA parses cleanly and finds nothing -- the first
    inspection run reported 36,081 regions and zero classes before this was
    fixed. A rehearsal against synthetic *standard* VIA had passed, which is
    exactly the trap: the rehearsal proved the code ran, not that the assumption
    about the format held.
    """
    if isinstance(region.get("class"), str):
        return normalise(region["class"])

    attributes = region.get("region_attributes") or {}
    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalise(value)
    return None


def _region_points(region: dict) -> tuple[list, list]:
    """Polygon points, from either annotation shape."""
    if "all_x" in region:
        return list(region.get("all_x") or []), list(region.get("all_y") or [])
    shape = region.get("shape_attributes") or {}
    return list(shape.get("all_points_x") or []), list(shape.get("all_points_y") or [])


def _is_polygon(region: dict) -> bool:
    if "all_x" in region:
        return True
    return (region.get("shape_attributes") or {}).get("name") == "polygon"


def inspect_via(annotation_file: Path) -> None:
    """Print what is actually in the file, before converting anything.

    VehiDE's class names will not match CLASSES exactly -- it has eight types,
    including `torn` and `lost parts`, which CarDD does not. Read this output and
    write VIA_TO_CLASS below accordingly; do not skip it. A silent mismatch here
    relabels every prediction.
    """
    via = json.loads(annotation_file.read_text())
    attribute_keys: Counter = Counter()
    class_names: Counter = Counter()
    shapes: Counter = Counter()

    for entry in via.values():
        for region in entry.get("regions", []):
            attribute_keys.update(k for k in region if k not in {"all_x", "all_y"})
            name = _region_class(region)
            if name:
                class_names[name] += 1
            shapes["polygon" if _is_polygon(region) else "other"] += 1

    print(f"images in file:       {len(via)}")
    print(f"region attribute keys: {dict(attribute_keys)}")
    print(f"shape types:           {dict(shapes)}")
    print()
    print("class -> instance count")
    for name, count in class_names.most_common():
        marker = "  <-- in CLASSES" if name in CLASSES else ""
        print(f"  {name:24s} {count:6d}{marker}")


#: VehiDE's class names are Vietnamese. Each is given with its literal meaning and
#: the English term from the paper's own class list, so the mapping can be checked
#: rather than trusted. Run inspect_via and confirm these are the names present
#: before training -- a wrong entry here relabels every prediction of that class.
#:
#: Counts are from data/vehide as measured by scripts/inspect_vehide.py.
VIA_TO_CLASS: dict[str, str | None] = {
    "tray_son": "scratch",         # tray son     -- paint scratch   14,647  40.6%
    "mop_lom": "dent",             # mop lom      -- dent             5,681  15.7%
    "rach": "torn",                # rach         -- torn             5,509  15.3%
    "mat_bo_phan": "missing_part",  # mat bo phan  -- lost part        2,818   7.8%
    "be_den": "lamp_broken",       # be den       -- broken lights    2,782   7.7%
    "thung": "punctured",          # thung        -- punctured        2,423   6.7%
    "vo_kinh": "glass_shatter",    # vo kinh      -- broken glass     2,221   6.2%
}


def convert_via(annotation_file: Path, image_dir: Path, out_dir: Path) -> int:
    """VIA polygons -> YOLO segmentation labels.

    Mirrors convert_coco: same output layout, same refusal to lose data quietly.
    """
    if not VIA_TO_CLASS:
        raise RuntimeError(
            "VIA_TO_CLASS is empty. Run inspect_via first and write the mapping "
            "explicitly -- an implicit one would relabel instances silently."
        )
    unknown = {k for k in VIA_TO_CLASS.values() if k is not None and k not in CLASSES}
    if unknown:
        raise RuntimeError(f"VIA_TO_CLASS maps to names not in CLASSES: {sorted(unknown)}")

    via = json.loads(annotation_file.read_text())
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)

    from PIL import Image

    written = 0
    unmapped: Counter = Counter()
    non_polygon: Counter = Counter()
    no_regions: list[str] = []
    missing_file: list[str] = []
    dropped_by_choice = 0

    for entry in via.values():
        filename = entry.get("name") or entry.get("filename")
        if not filename:
            continue
        source = image_dir / filename
        if not source.is_file():
            missing_file.append(str(filename))
            continue

        with Image.open(source) as image:
            width, height = image.size

        lines = []
        for region in entry.get("regions", []):
            if not _is_polygon(region):
                # VIA also writes rect/circle/ellipse. A box is not a mask, and
                # area_ratio is computed from the mask -- see vehicle_yolo.py.
                shape_name = (region.get("shape_attributes") or {}).get("name", "?")
                non_polygon[shape_name] += 1
                continue

            raw = _region_class(region)
            if raw not in VIA_TO_CLASS:
                unmapped[raw] += 1
                continue
            target = VIA_TO_CLASS[raw]
            if target is None:
                dropped_by_choice += 1
                continue

            xs, ys = _region_points(region)
            if len(xs) < 3 or len(xs) != len(ys):
                continue

            coords = []
            for x, y in zip(xs, ys):
                coords.append(min(1.0, max(0.0, x / width)))
                coords.append(min(1.0, max(0.0, y / height)))
            lines.append(
                str(CLASSES.index(target)) + " " + " ".join(f"{c:.6f}" for c in coords)
            )

        if not lines:
            no_regions.append(str(filename))
            continue

        (out_dir / "labels" / f"{source.stem}.txt").write_text("\n".join(lines))
        (out_dir / "images" / source.name).write_bytes(source.read_bytes())
        written += 1

    total = len(via)
    print(f"converted {written}/{total} images")
    if dropped_by_choice:
        print(f"  {dropped_by_choice} instance(s) dropped by VIA_TO_CLASS (mapped to None)")
    if unmapped:
        print(f"  UNMAPPED classes -- these instances were lost: {dict(unmapped)}")
    if non_polygon:
        print(f"  non-polygon shapes skipped: {dict(non_polygon)}")
    for label, names in (("no usable polygon", no_regions), ("image missing", missing_file)):
        if names:
            print(f"  {len(names)} skipped -- {label}: {', '.join(names[:5])}"
                  + (" ..." if len(names) > 5 else ""))
    if total and (total - written) > total * 0.05:
        print(f"  WARNING: {(total - written) / total:.0%} of images dropped. Investigate "
              "before training -- large enough to change the class balance.")

    return written


print("Run inspect_via first, fill in VIA_TO_CLASS, then convert_via.")


### Check the class names before converting

The mapping below is written out term by term with counts, so it can be checked
rather than trusted. Run this and confirm the names match.


In [ ]:
print("=== TRAIN ===")
inspect_via(TRAIN_ANNOS)
print()
print("=== VAL ===")
inspect_via(VAL_ANNOS)


## 4. Convert both splits

VehiDE's validation set becomes our **test** set — held out entirely, never
trained on. Its training set is split into train and val for early stopping.

This is deliberate. Pooling everything and reshuffling would give a slightly
larger training set and a number nobody can compare against the dataset's own
published results.


In [ ]:
CONVERTED_TRAIN = WORK / "converted_train"
CONVERTED_TEST = WORK / "converted_test"

n_train = convert_via(TRAIN_ANNOS, TRAIN_IMAGES, CONVERTED_TRAIN)
print()
n_test = convert_via(VAL_ANNOS, VAL_IMAGES, CONVERTED_TEST)
print()
print(f"converted: {n_train:,} for train/val, {n_test:,} held out for test")


In [ ]:
import hashlib
import shutil


def split_dataset(
    converted: Path, target: Path, train_ratio: float, val_ratio: float
) -> dict[str, int]:
    stems = sorted(p.stem for p in (converted / "images").iterdir())
    rng = random.Random(SEED)
    rng.shuffle(stems)

    n = len(stems)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)
    splits = {
        "train": stems[:train_end],
        "val": stems[train_end:val_end],
        "test": stems[val_end:],
    }

    for name, members in splits.items():
        for kind in ("images", "labels"):
            (target / name / kind).mkdir(parents=True, exist_ok=True)
        for stem in members:
            for source in (converted / "images").glob(f"{stem}.*"):
                shutil.copy2(source, target / name / "images" / source.name)
            label = converted / "labels" / f"{stem}.txt"
            if label.is_file():
                shutil.copy2(label, target / name / "labels" / label.name)

    # A fingerprint of the split itself. Record it with the metrics: if it ever
    # differs, the numbers are not comparable to the published ones.
    digest = hashlib.sha256(
        "|".join(f"{k}:{','.join(v)}" for k, v in sorted(splits.items())).encode()
    ).hexdigest()[:16]
    print(f"split fingerprint: {digest}")

    return {name: len(members) for name, members in splits.items()}


print("Run after conversion.")

## 5. Assemble the dataset

`split_dataset` divides VehiDE's training images into train and val; the test
directory is VehiDE's validation set, copied across untouched.


In [ ]:
import shutil

DATASET = WORK / "vehide_yolo"
if DATASET.exists():
    shutil.rmtree(DATASET)

# 82/18 of VehiDE's train split, so val is roughly the size of the test set.
counts = split_dataset(CONVERTED_TRAIN, DATASET, train_ratio=0.82, val_ratio=0.18)

# split_dataset assigns the remainder to `test`, and integer rounding leaves one:
# 11,621 images produce 9,529 + 2,091 = 11,620. That image must not stay in test
# -- "the test set was never trained on" is either true or it is not -- and it
# must not be thrown away either, which is what deleting the directory did on
# the first attempt. Move it into train.
leftover = DATASET / "test"
if leftover.exists():
    moved = 0
    for kind in ("images", "labels"):
        for source in (leftover / kind).iterdir():
            shutil.move(str(source), str(DATASET / "train" / kind / source.name))
            moved += kind == "images"
    shutil.rmtree(leftover)
    if moved:
        print(f"moved {moved} rounding-remainder image(s) from test into train")
    counts["train"] += moved
counts.pop("test", None)

# VehiDE's own validation set, held out as test.
for kind in ("images", "labels"):
    target = DATASET / "test" / kind
    target.mkdir(parents=True, exist_ok=True)
    for source in (CONVERTED_TEST / kind).iterdir():
        shutil.copy2(source, target / source.name)

counts["test"] = len(list((DATASET / "test" / "images").iterdir()))
print(counts)
assert counts["test"] > 0, "the test split is empty"


In [ ]:
data_yaml = WORK / "vehide.yaml"
data_yaml.write_text(
    f"""path: {DATASET}
train: train/images
val: val/images
test: test/images

# Order matters: it is the contract with vehicle_yolo.py::VEHIDE_CLASSES.
names:
"""
    + "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASSES))
)
print(data_yaml.read_text())


### Is there already a trained checkpoint?

**If you have run this notebook before, do not train 640 px again.** A finished
`best.pt` is a complete answer to the first six sections, and re-deriving it costs
the same hours it cost the first time -- hours that section 9 needs and that
Kaggle's 9-hour session will not give you twice.

To reuse one: **+ Add Input → Datasets → New Dataset**, upload your `best.pt`, and
attach it. The cell below finds it wherever Kaggle mounts it, and sections 6 and 7
become no-ops.

Nothing is uploaded? Then this reports so and the notebook trains from scratch, as
it did the first time.


In [ ]:
UPLOADED_640 = None

# Anything under /kaggle/input is read-only and came from a dataset the user
# attached. A checkpoint found there was trained in an earlier session.
for root in (Path("/kaggle/input"), Path("/content/drive/MyDrive"), WORK.parent):
    if not root.exists():
        continue
    for candidate in sorted(root.rglob("*.pt")):
        # Skip the pretrained COCO weights, which are also a .pt and are not this.
        if candidate.name in {"yolo11s-seg.pt", "yolo11n-seg.pt", MODEL}:
            continue
        if RUNS in candidate.parents:
            continue  # produced by this session, handled by section 6 itself
        UPLOADED_640 = candidate
        break
    if UPLOADED_640:
        break

if UPLOADED_640 is None:
    # Kaggle normally unpacks an uploaded archive, but it is not guaranteed, and
    # the failure is silent in the worst way: no .pt found means "train from
    # scratch", which is the six hours this whole section exists to avoid.
    import zipfile

    for root in (Path("/kaggle/input"), Path("/content/drive/MyDrive")):
        if not root.exists():
            continue
        for archive in sorted(root.rglob("*.zip")):
            try:
                with zipfile.ZipFile(archive) as bundle:
                    members = [n for n in bundle.namelist() if n.endswith(".pt")]
                    if not members:
                        continue
                    unpacked = WORK / "uploaded"
                    unpacked.mkdir(parents=True, exist_ok=True)
                    bundle.extract(members[0], unpacked)
                    UPLOADED_640 = unpacked / members[0]
                    print(f"unpacked {members[0]} from {archive.name}")
                    break
            except zipfile.BadZipFile:
                continue
        if UPLOADED_640:
            break

if UPLOADED_640:
    from ultralytics import YOLO

    probe_ckpt = YOLO(str(UPLOADED_640))
    epoch = probe_ckpt.ckpt.get("epoch", "?") if getattr(probe_ckpt, "ckpt", None) else "?"
    names = probe_ckpt.names

    print(f"found: {UPLOADED_640}")
    print(f"  size:    {UPLOADED_640.stat().st_size / 1e6:.1f} MB")
    print(f"  epoch:   {epoch}")
    print(f"  classes: {len(names)} -- {list(names.values())[:4]} ...")
    print()

    if len(names) != len(CLASSES):
        print(f"REFUSING it: {len(names)} classes, this dataset has {len(CLASSES)}.")
        print("That is the COCO-pretrained model, not a VehiDE-trained one.")
        print("Sections 6 and 7 will train from scratch.")
        UPLOADED_640 = None
    else:
        print("Sections 6 and 7 will SKIP training and use this checkpoint.")
        print("Section 9 will warm-start from it.")
else:
    print("no uploaded checkpoint -- sections 6 and 7 will train 640 px from scratch")


### First, measure the pace

Two epochs, to find out what this session actually costs before committing to a
hundred of them. The estimate below is measured on this GPU with this dataset,
not looked up.

If the projection exceeds the session limit — 9 h on Kaggle, 12 h on Colab —
lower `EPOCHS`, or raise `BATCH` if the GPU has memory to spare. Early stopping
usually ends the run well before the ceiling anyway.


In [ ]:
import time

from ultralytics import YOLO

if UPLOADED_640:
    print("skipping the pace probe -- not training 640 px this session")
    per_epoch = None
else:
    probe = YOLO(MODEL)
    start = time.perf_counter()
    probe.train(
        data=str(data_yaml), epochs=2, imgsz=IMGSZ, batch=BATCH, seed=SEED,
        project=str(WORK / "probe"), name="pace", exist_ok=True, val=False, plots=False,
    )
    per_epoch = (time.perf_counter() - start) / 2

    print()
    print(f"measured: {per_epoch / 60:.1f} min/epoch")
    print(f"  {EPOCHS} epochs -> {per_epoch * EPOCHS / 3600:.1f} h")
    print(f"  early stop around 60 -> {per_epoch * 60 / 3600:.1f} h")

    limit = 9 if PLATFORM == "kaggle" else 12
    if per_epoch * EPOCHS / 3600 > limit:
        print()
        print(f"WARNING: a full {EPOCHS}-epoch run exceeds this platform's {limit} h session.")
        print("Lower EPOCHS, or rely on patience=20 stopping earlier -- and note that")
        print("`resume=True` on last.pt picks up where a disconnect left off.")
        print()
        print("Running section 9 as well in one session will not fit. Train 640 px now,")
        print("download best.pt, and start a fresh session with it attached as a dataset.")


## 6. Train

Output goes straight to Drive. Ultralytics writes `last.pt` every epoch, so a
disconnect costs one epoch — resume with `YOLO(RUNS/'vehide_seg/weights/last.pt')`
and `model.train(resume=True)`.


In [ ]:
from ultralytics import YOLO

if UPLOADED_640:
    print(f"skipping the 640 px run -- using {UPLOADED_640.name}")
    print("Delete the attached dataset and re-run if you want to train it again.")
    BEST_640 = UPLOADED_640
    results = None
else:
    model = YOLO(MODEL)

    results = model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        deterministic=True,
        patience=20,
        project=str(RUNS),      # Drive, not /content -- survives a dead session
        name="vehide_seg",
        exist_ok=True,
        # Damage is small and often low-contrast; heavy colour jitter hurts more than
        # it helps. Keep the defaults conservative and let the report say what happened.
        hsv_v=0.3,
        degrees=5.0,
        fliplr=0.5,
        mosaic=1.0,
        close_mosaic=10,
    )
    BEST_640 = RUNS / "vehide_seg" / "weights" / "best.pt"


## 7. Evaluate on the held-out test split

This is VehiDE's own validation set, which the model has never seen.

**Paste the table into README section 7.3 verbatim, including the bad rows.**
`scratch` is 40% of the data and the other six share the rest; expect that to
show, and publish it anyway.


In [ ]:
best = BEST_640
trained = YOLO(str(best))

metrics = trained.val(data=str(data_yaml), split="test", imgsz=IMGSZ)

print("| Class | mAP@50 | mAP@50-95 | Precision | Recall |")
print("|---|---|---|---|---|")
for i, name in enumerate(CLASSES):
    p, r, m50, m5095 = metrics.seg.class_result(i)
    print(f"| {name} | {m50:.3f} | {m5095:.3f} | {p:.3f} | {r:.3f} |")
print(
    f"| **all** | {metrics.seg.map50:.3f} | {metrics.seg.map:.3f} | "
    f"{metrics.seg.mp:.3f} | {metrics.seg.mr:.3f} |"
)


## 8. Export

Download `best.pt` and put it at `backend/weights/vehide_yolo_seg.pt`.

Record the SHA-256 and the split fingerprint alongside the metrics. If either
ever differs, the numbers are not comparable to the published ones.


In [ ]:
import hashlib

digest = hashlib.sha256(best.read_bytes()).hexdigest()
print(f"vehide_yolo_seg.pt")
print(f"  sha256: {digest}")
print(f"  size:   {best.stat().st_size / 1e6:.1f} MB")
print(f"  path:   {best}")
print()
print("Copy to backend/weights/vehide_yolo_seg.pt, then restart the API.")
print("The vehicle domain starts measuring -- no code change.")

if PLATFORM == "colab":
    from google.colab import files

    files.download(str(best))
else:
    print()
    print("On Kaggle: Save Version (Quick Save), then download best.pt from the")
    print("notebook's Output tab.")


---

## 9. Optional: a second pass at 960 px, for thin damage

**Run this only after reading the per-class table from section 7.**

The 640 px run scored `scratch` at mAP@50 **0.239** and `dent` at **0.244**,
against `glass_shatter` at **0.782** — with `scratch` holding six times the
training data of `glass_shatter`. More data did not help, which points at the
damage rather than the dataset: a scratch is thin and low-contrast, and
downscaling a 1.7-megapixel photograph to 640 px is exactly the operation that
removes thin, low-contrast detail.

That is a hypothesis, not a finding. This section tests it.

**It continues from the 640 px checkpoint rather than starting over.** Weights
that already know what a dent looks like do not need to relearn it at a higher
resolution, and a fresh 100-epoch run at 960 px would take roughly twice as long
as the first — past Kaggle's 9-hour session. Forty epochs from a warm start fits
comfortably.

**The split is unchanged.** Same seed, same files, same held-out test set, so
the two tables are comparable. If the fingerprint printed in section 5 differs
from the first run's, stop — the comparison is meaningless and the cause is a
changed dataset, not a changed resolution.

**Batch drops to 8.** 960 px activations are roughly 2.25x the memory of 640 px;
16 will not fit on a single T4.

If `scratch` does not move, that is a real result and worth publishing as one:
it would mean the ceiling is annotation quality rather than resolution, and no
amount of retraining at higher resolution will fix it.


In [ ]:
IMGSZ_HI = 960
EPOCHS_HI = 40
BATCH_HI = 8

warm_start = BEST_640
assert warm_start.exists(), f"no 640 px checkpoint at {warm_start} -- run section 6"

print(f"warm-starting from {warm_start}")

hi = YOLO(str(warm_start))

results_hi = hi.train(
    data=str(data_yaml),
    epochs=EPOCHS_HI,
    imgsz=IMGSZ_HI,
    batch=BATCH_HI,
    seed=SEED,
    deterministic=True,
    patience=10,
    project=str(RUNS),
    name="vehide_seg_960",
    exist_ok=True,
    # Same augmentation as the first pass. Changing two things at once would
    # leave the comparison unable to say which one moved the number.
    hsv_v=0.3,
    degrees=5.0,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
)


In [ ]:
best_hi = RUNS / "vehide_seg_960" / "weights" / "best.pt"
trained_hi = YOLO(str(best_hi))

metrics_hi = trained_hi.val(data=str(data_yaml), split="test", imgsz=IMGSZ_HI)

# The 640 px figures, from the run this notebook already produced. Written out
# rather than recomputed so the comparison survives a fresh session.
BASELINE_640 = {
    "dent": 0.244,
    "glass_shatter": 0.782,
    "lamp_broken": 0.479,
    "missing_part": 0.649,
    "punctured": 0.458,
    "scratch": 0.239,
    "torn": 0.285,
}

print("| Class | mAP@50 @640 | mAP@50 @960 | change |")
print("|---|---|---|---|")
for index, name in enumerate(CLASSES):
    _, _, m50_hi, _ = metrics_hi.seg.class_result(index)
    before = BASELINE_640[name]
    delta = m50_hi - before
    arrow = "up" if delta > 0.01 else ("down" if delta < -0.01 else "flat")
    print(f"| {name} | {before:.3f} | {m50_hi:.3f} | {delta:+.3f} {arrow} |")
print(f"| **all** | 0.448 | {metrics_hi.seg.map50:.3f} | {metrics_hi.seg.map50 - 0.448:+.3f} |")

print()
print("Inference cost roughly doubles at 960 px. Section 7.3 of the README puts")
print("the 640 px specialist at 113 ms on the production CPU; if the gain here is")
print("small, 640 is the better trade and this run is a recorded negative result.")


In [ ]:
import hashlib

digest_hi = hashlib.sha256(best_hi.read_bytes()).hexdigest()
print("vehide_yolo_seg_960.pt")
print(f"  sha256: {digest_hi}")
print(f"  size:   {best_hi.stat().st_size / 1e6:.1f} MB")
print()
print("Only adopt this if the table above justifies the doubled inference cost.")
print("Swapping it in means changing IMGSZ in the specialist as well -- a model")
print("trained at 960 and run at 640 performs worse than either done consistently.")
